# Chapter 3: a mass balance on one page

The book asks for one table: what went into each stage, what came out and where the difference went. This notebook builds it from the minimum dataset (one row per batch per stage) and flags any stage where the balance does not close.

A month here means the beer cast in that month, followed to dispatch. Brewhouse losses are counted in kilograms of extract (chapter 5), so the hL balance starts at cast.

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/minimum_dataset.csv", parse_dates=["date_in", "date_out"])
cast_dates = df[df.stage == "fermentation"].set_index("batch_id")["date_in"]
df["month"] = df.batch_id.map(cast_dates).dt.to_period("M")
df.head(8)

## The stage table for one month

January is the month printed in chapter 3.2. Change `MONTH` to see February or March.

In [ ]:
MONTH = "2026-01"


def stage_table(d):
    s = d.groupby("stage")[["volume_in_hl", "volume_out_hl"]].sum()
    cast = s.loc["fermentation", "volume_in_hl"]
    rows = [
        ("Cellar: fermentation, maturation, transfers, dry hop", cast, s.loc["filter", "volume_in_hl"]),
        ("Filtration and bright beer tanks", s.loc["filter", "volume_in_hl"], s.loc["filter", "volume_out_hl"]),
        ("Packaging: fill giveaway, rejects, line flush", s.loc["package", "volume_in_hl"], s.loc["package", "volume_out_hl"]),
        ("Not explained (packed less dispatched)", s.loc["dispatch", "volume_in_hl"], s.loc["dispatch", "volume_out_hl"]),
    ]
    t = pd.DataFrame(rows, columns=["Stage", "In (hL)", "Out (hL)"])
    t["Loss (hL)"] = t["In (hL)"] - t["Out (hL)"]
    t["Loss, % of cast"] = 100 * t["Loss (hL)"] / cast
    total = pd.DataFrame([["Total", cast, t["Out (hL)"].iloc[-1], t["Loss (hL)"].sum(), 100 * t["Loss (hL)"].sum() / cast]], columns=t.columns)
    return pd.concat([t, total], ignore_index=True).round(2)


table = stage_table(df[df.month == MONTH])
table

## Flags: where does the balance not close?

Two checks. First, batch by batch: each stage should start with the volume the stage before it ended with. A transfer line holds a little beer, so a small gap is normal. A gap above the tolerance (0.7 percent, chapter 4.4) is flagged as a meter or a booking to check. Second, month by month: the part no stage explains, as a share of cast, against the same tolerance.

In [ ]:
TOLERANCE_PCT = 0.7
ORDER = ["fermentation", "conditioning", "filter", "package", "dispatch"]

flags = []
for batch, g in df[df.stage != "cast"].groupby("batch_id"):
    g = g.set_index("stage").reindex(ORDER)
    for prev, nxt in zip(ORDER, ORDER[1:]):
        out_prev, in_next = g.loc[prev, "volume_out_hl"], g.loc[nxt, "volume_in_hl"]
        if pd.isna(out_prev) or pd.isna(in_next):
            flags.append((batch, f"{prev} to {nxt}", "missing row"))
            continue
        gap_pct = 100 * (out_prev - in_next) / out_prev
        if abs(gap_pct) > TOLERANCE_PCT:
            flags.append((batch, f"{prev} to {nxt}", f"gap {out_prev - in_next:.1f} hL ({gap_pct:.2f} percent)"))
print("Batch flags:")
print(pd.DataFrame(flags, columns=["batch_id", "step", "flag"]).to_string(index=False) if flags else "none")

for month, g in df.groupby("month"):
    t = stage_table(g)
    cast = t["In (hL)"].iloc[-1]
    unexplained_pct = 100 * t.loc[3, "Loss (hL)"] / cast
    verdict = "closes" if unexplained_pct <= TOLERANCE_PCT else "DOES NOT CLOSE: fix the measurement before pricing"
    print(f"{month}: cast {cast:,.0f} hL, not explained {unexplained_pct:.2f} percent: {verdict}")

## The month in money

Cellar beer is valued at the replacement cost of USD 30 a hL, beer after the filter at 32 (chapter 2.3). The peak season upper bound is 110 and 112. Never add the two columns.

In [ ]:
cellar = table.loc[0, "Loss (hL)"]
after_filter = table.loc[1:3, "Loss (hL)"].sum()
replacement = cellar * 30 + after_filter * 32
upper = cellar * 110 + after_filter * 112
print(f"Gap {cellar + after_filter:,.0f} hL: replacement cost USD {replacement:,.0f} a month, peak season upper bound USD {upper:,.0f}")

normal = 0.0045 * table.loc[4, "In (hL)"]
excess = table.loc[1, "Loss (hL)"] - round(normal)
print(f"Filtration normal {normal:.0f} hL, actual {table.loc[1, 'Loss (hL)']:.0f} hL, excess {excess:.0f} hL: USD {excess * 32:,.0f} (upper bound USD {excess * 112:,.0f})")

In [ ]:
t = table.iloc[:4]
fig, ax = plt.subplots(figsize=(7, 3.5))
run = table.loc[4, "In (hL)"]
ax.bar(0, run, color="#0b4f45")
for i, loss in enumerate(t["Loss (hL)"], start=1):
    ax.bar(i, loss, bottom=run - loss, color="#b23a2e" if i == 4 else "#2f7f72")
    run -= loss
ax.bar(5, run, color="#0b4f45")
ax.set_xticks(range(6), ["Cast", "Cellar", "Filter and\nbright tank", "Packaging", "Not\nexplained", "Dispatched"])
ax.set_ylim(8200, 8750)
ax.set_ylabel("hL of beer (axis starts at 8,200)")
ax.set_title(f"Mass balance, beer cast in {MONTH} (synthetic)")
plt.tight_layout()
plt.show()

**Your own data.** Replace `minimum_dataset.csv` with your own rows in the same columns. The notebook runs unchanged.